<a href="https://colab.research.google.com/github/YefridC09/ST-554-Project1-Template/blob/main/Task1/st554-project1-task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Title: ST-554 Project 1 Task 1 \
Author: Stephen Griggs \
Date: 2/11/2026

In [1]:
!pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepo
import numpy as np
from math import sqrt

Remove any observations where the C6H6(GT) or CO(GT) are -200 as these represent missing values (which
we’ll ignore).

In [3]:
air_quality = fetch_ucirepo(id=360)
air_quality = air_quality.data.features
air_quality = air_quality[
    (air_quality["C6H6(GT)"] != -200) &
    (air_quality["CO(GT)"] != -200)
]

Response variable: C6H6(GT) \
Loss function
$$
  \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - c)^2}
$$

1. One where we don't consider any data other than the y's. That is, c is going to just be a constant that minimizes the function.
*Note: the calculus based answer for this comes out to be the sample mean, ȳ

2. One where we consider a linear equation (the SLR model) of one other numeric variable.
* Relating to the above, for each observation (i) our prediction is given by $c_i = b_0 + b_1 x_i$.
* Here xi is one of the other numeric variables from our data set. We’ll use the PT08.S1(CO)
variable.
* Note: the calculus based answer for this comes out to be the usual simple linear regression
estimates, which you can find using scipy.stats as done in the Fitting and Evaluating SLR
Models notes!

Using a Grid Search Algorithm

You'll be implementing a grid search to find the optimal value of $c$ based off of our data set.

Just $y$: Pseudo code for using just the $y$'s and no other variables:

1. Create a grid of values for $c$. Look at the first and third quartiles for the $\mathrm{C6H6(GT)}$ variable to consider reasonable values for $c$.

2. Create a squared error loss function that takes in $y$ and $c$ that outputs $(y - c)^2$.

3. Create a root mean squared error (objective) function that takes in $y$ and $c$ that outputs $\sqrt{ \frac{1}{n} \sum_{i=1}^{n} (y_i - c)^2 }$ (You can put step 2 and 3 into one function if you want.)

4. Use a list comprehensive to loop over the grid of $c$ values, finding the RMSE for each value of $c$.

5. Determine which value of $c$ gives the optimal (smallest) RMSE.

6. Report that as the prediction!

7. Wrap the above into a function that takes in a column of data and outputs the value of $c$.

- Run your algorithm on the $\mathrm{C6H6(GT)}$ variable and determine the \textbf{optimal} constant prediction.

- Just to make sure your algorithm generalizes, run it using the $\mathrm{PT08.S1(CO)}$ variable as the response (this shouldn't involve any new functions, just new function calls!). Be sure to adjust your grid accordingly (use the quantiles of this variable)!

In [4]:
# root mean square error function
def calc_rmse(response, c):
    return sqrt(1/len(response) * sum((response-c)**2))

def find_best_c(response, num_points=100):
    # q1 and q3 of response column
    q1, q3 = response.quantile([0.25, 0.75])
    grid = np.linspace(q1, q3, num_points)

    # loop over grid of c values finding rmse for each
    rmse_vals = [calc_rmse(response, c) for c in grid]
    paired = list(zip(grid, rmse_vals))

    # determine which value of c gives the smallest rmse
    best_c, min_rmse = min(paired, key=lambda x: x[1])

    # report best value of c as a prediction
    print(f"The best value of c is {best_c} with an RMSE of {min_rmse}")

    # return best value of c
    return best_c

In [5]:
find_best_c(air_quality["C6H6(GT)"])
find_best_c(air_quality["PT08.S1(CO)"])
np.mean(air_quality["C6H6(GT)"])
np.mean(air_quality["PT08.S1(CO)"])

The best value of c is 10.282828282828284 with an RMSE of 7.440564471926774
The best value of c is 1109.6363636363635 with an RMSE of 218.6684818310563


np.float64(1110.5807461873637)

Using y and another numeric variable x: \
Next, you'll implement the grid search to find the optimal pair of values for b0 and b1 using PT08.S1(CO) as your x variable and C6H6(GT) as your y variable. The pseudo code is very similar to that above, but your grid now has two-dimensions!
* You'll need to populate a grid of b0 and b1 values that you want to consider.
  * Use b0 values from -25 to -15 with increments of 0.1.
  * Use b1 values from -5 to 5 with increments of 0.01
* Report your optimal b0 and b1 combination.
* Create a function that takes in an x and and y column and outputs the optimal values.
* Then use these values to predict a new C6H6(GT) for a PT08.S1(CO) of 946, 1075, and 1246.

In [6]:
def find_best_c(response, predictor):
    # grid for predictor and response (beta0 and beta1)
    beta0_grid = np.arange(-25, -15, 0.1)
    beta1_grid = np.arange(-5, 5, 0.01)

    # loop over grid of c values finding rmse for each variable
    rmse_grid = [(b0, b1, calc_rmse(response, b0 + b1 * predictor))
           for b0 in beta0_grid
           for b1 in beta1_grid]

    # determine which vector c gives the smallest rmse
    best_b0, best_b1, min_rmse = min(rmse_grid, key=lambda x: x[2])

    # report best vector c as a prediction
    print(f"The best values for beta0 and beta1 are ({best_b0}, {best_b1} with an RMSE of {min_rmse}")

    # return best values of beta0 and beta1
    return best_b0, best_b1

In [7]:
# Find best coefficients using your data
beta0, beta1 = find_best_c(air_quality["C6H6(GT)"], air_quality["PT08.S1(CO)"])

# Predict C6H6(GT) for new PT08.S1(CO) values
new_values = np.array([946, 1075, 1246])
predictions = beta0 + beta1 * new_values

print(predictions)

The best values for beta0 and beta1 are (-22.99999999999997, 0.02999999999989278 with an RMSE of 3.5429053908306787
[ 5.38  9.25 14.38]
